In [ ]:
# HARD RESET
!rm -rf /content/UIDAI-Hackathon-2026
%cd /content

# CLONE GITHUB REPO
!git clone https://github.com/Romit-M/UIDAI-Hackathon-2026.git
%cd UIDAI-Hackathon-2026


/content
Cloning into 'UIDAI-Hackathon-2026'...
remote: Enumerating objects: 310, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 310 (delta 1), reused 3 (delta 0), pack-reused 304 (from 2)
Receiving objects: 100% (310/310), 418.18 MiB | 16.08 MiB/s, done.
Resolving deltas: 100% (163/163), done.
Updating files: 100% (29/29), done.
/content/UIDAI-Hackathon-2026


### **Imports & Helpers**

In [ ]:
# IMPORTS
!pip install rapidfuzz
from rapidfuzz import fuzz, process

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# HELPER FUNCTIONS

# ======================
# Get file names
# ======================
def get_filename(category, r):
    return f"api_data_aadhar_{category}_{r}.csv"


# ======================
# Data loader
# ======================
def load_data(input_csv):
    df = pd.read_csv(input_csv)

    return df


# ======================
# Cached fuzzy matcher
# ======================

def fuzzy_match(value, choices, threshold=90):
    key = (value, tuple(choices))
    if key in cache:
        return cache[key]

    match = process.extractOne(value, choices, scorer=fuzz.token_sort_ratio)
    result = match[0] if match and match[1] >= threshold else None
    cache[key] = result
    return result


### **1. Dataset Alignment (implemented on local system due to RAM constraints)**

#### **Vertical Stacking (Pre-Merge)**

In [ ]:
# # 1. Load both datasets for each category

# # BIOMETRIC
# FILE_PATH = "data/processed/api_data_aadhar_biometric/"
# bio_p1 = pd.read_csv(f'{FILE_PATH}api_data_aadhar_biometric_0_500000_cleaned.csv')
# bio_p2 = pd.read_csv(f'{FILE_PATH}api_data_aadhar_biometric_500000_1000000_cleaned.csv')
# df_biometric = pd.concat([bio_p1, bio_p2], axis=0, ignore_index=True)

# # DEMOGRAPHIC
# FILE_PATH = "data/processed/api_data_aadhar_demographic/"
# demo_p1 = pd.read_csv(f'{FILE_PATH}api_data_aadhar_demographic_0_500000_cleaned.csv')
# demo_p2 = pd.read_csv(f'{FILE_PATH}api_data_aadhar_demographic_500000_1000000_cleaned.csv')
# df_demographic = pd.concat([demo_p1, demo_p2], axis=0, ignore_index=True)

# # ENROLMENT
# FILE_PATH = "data/processed/api_data_aadhar_enrolment/"
# enrol_p1 = pd.read_csv(f'{FILE_PATH}api_data_aadhar_enrolment_0_500000_cleaned.csv')
# enrol_p2 = pd.read_csv(f'{FILE_PATH}api_data_aadhar_enrolment_500000_1000000_cleaned.csv')
# df_enrolment = pd.concat([enrol_p1, enrol_p2], axis=0, ignore_index=True)


# # 2. Remove duplicates by key-based aggregation
# group_cols = ['date', 'state', 'district', 'pincode']

# df_biometric = df_biometric.groupby(group_cols).sum().reset_index()
# df_demographic = df_demographic.groupby(group_cols).sum().reset_index()
# df_enrolment = df_enrolment.groupby(group_cols).sum().reset_index()


# print("Vertical Stacking Complete")

/tmp/ipython-input-3514743241.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  bio_p1 = pd.read_csv(f'{FILE_PATH}api_data_aadhar_biometric_0_500000_cleaned.csv')
/tmp/ipython-input-3514743241.py:11: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  demo_p1 = pd.read_csv(f'{FILE_PATH}api_data_aadhar_demographic_0_500000_cleaned.csv')
/tmp/ipython-input-3514743241.py:17: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  enrol_p1 = pd.read_csv(f'{FILE_PATH}api_data_aadhar_enrolment_0_500000_cleaned.csv')


Vertical Stacking Complete


#### **Horizontal Merge (Master Merge)**

In [ ]:
# # 3. Standardize column names
# df_biometric = df_biometric.rename(columns={'bio_age_5_17': 'bio_child', 'bio_age_17_': 'bio_adult'})
# df_demographic = df_demographic.rename(columns={'demo_age_5_17': 'demo_child', 'demo_age_17_': 'demo_adult'})
# df_enrolment = df_enrolment.rename(columns={'age_0_5': 'enrol_infant', 'age_5_17': 'enrol_child', 'age_18_greater': 'enrol_adult'})


# # 4. Standardize date format
# for df in [df_biometric, df_demographic, df_enrolment]:
#     df['date'] = pd.to_datetime(df['date'], dayfirst=True)


# # 5. Horizontal Merge (Combine all three datasets)
# df_master = pd.merge(df_biometric, df_demographic, on=['date', 'state', 'district', 'pincode', 'valid_flag', 'pin_prefix'], how='outer')
# df_master = pd.merge(df_master, df_enrolment, on=['date', 'state', 'district', 'pincode', 'valid_flag', 'pin_prefix'], how='outer')


# # =====================
# # FEATURE ENGINEERING
# # =====================

# # Remove "Not Valid" entries
# df_master = df_master[df_master['valid_flag'] != 'Not Valid']

# # Remove the 'valid_flag' column
# df_master = df_master.drop(columns=['valid_flag'])

# # Replace NaN with 0
# df_master = df_master.fillna(0)

# # Saving master dataset (with compression)
# df_master.to_parquet("master_dataset.parquet", index=False)       # smaller file size
# # df_master.to_csv("data/processed/master_dataset.csv", index=False, compression='gzip')


In [ ]:
# Load master dataset
df_master = pd.read_parquet("data/processed/master_dataset.parquet")

### **2. Derived Metrics Calculation**

#### **2.1 Analytical Feature Engineering for Performance Metrics**

In [ ]:
# A. Total Volume Metrics
df_master['total_updates'] = (df_master['bio_child'] + df_master['bio_adult'] +
                               df_master['demo_child'] + df_master['demo_adult'])

df_master['total_enrolments'] = (df_master['enrol_infant'] + df_master['enrol_child'] +
                                  df_master['enrol_adult'])


# B. The Aadhaar Stress Index (ASI)
# High ASI = System is busy maintaining old IDs. Low ASI = System is growing.
df_master['ASI'] = df_master['total_updates'] / (df_master['total_enrolments'] + 1)

# C. Child Transition Velocity
# Ratio of children updating biometrics vs those enrolling.
# Highlights districts with strong school-based compliance.
df_master['child_update_velocity'] = df_master['bio_child'] / (df_master['enrol_child'] + 1)

# D. Adult Friction Index
# Specifically looking at 18+ biometric updates (often signaling manual labor/aging)
df_master['adult_friction'] = df_master['bio_adult'] / (df_master['enrol_adult'] + 1)


#### **2.2 Census Integration**

In [ ]:
# 1. Load Census Data
df_census = pd.read_csv("data/external/census_data.csv")

# # 2. Join with Master Data
# df_master['district'] = df_master['District'].str.upper().str.strip()
# df_census['district'] = df_census['District_Name'].str.upper().str.strip()

# 2. AGGREGATE CENSUS DATA FIRST
# We sum the population of all pincodes to get one value per district
df_census_district = df_census.groupby('district')['total_population'].sum().reset_index()

# 3. Join with Master Data
# Now, one district in Master matches exactly one row in df_census_district
df_master = pd.merge(
    df_master,
    df_census_district,
    on='district',
    how='left'
)

# 4. Calculate Per-Capita Stress
# More accurate than raw ASI
df_master['updates_per_capita'] = df_master['total_updates'] / df_master['total_population']


### **Saving progress to GitHub repository**

In [ ]:
# PUSH CLEANED FILES TO REPO

from google.colab import userdata
pat = userdata.get('GitHubAccessToken')

!git config --global user.name "Romit-M"
!git config --global user.email "romitrmaity@gmail.com"

!git add .
!git commit -m "Added cleaned data files"
!git push https://{pat}@github.com/Romit-M/UIDAI-Hackathon-2026.git
